# 02 — Anomaly Detection Comparison
Compares the three detection methods DataPilot uses and validates their
accuracy on known anomalies embedded in the sample datasets.

**Real-time integration**: each detected anomaly fires an `anomaly:detected`
Socket.IO event immediately. This notebook measures:
- False positive rates per method (too many = noisy UI ticker)
- Detection latency (time from dataset load to first `anomaly:detected` event)
- Consensus detection improvement (both methods agree = higher confidence)
- Whether the severity classifications match human intuition


In [9]:
import sys, time
sys.path.insert(0, '..')
import polars as pl
import pandas as pd
import numpy as np

retail = pl.read_csv('datasets/sample_retail_sales.csv')
iot    = pl.read_parquet('datasets/sample_iot_timeseries.parquet')

# Known ground-truth anomalies we embedded
KNOWN_ANOMALIES = {
    'retail': {
        'revenue': [(-120.5, 'negative'), (-45.0, 'negative'), (-890.0, 'negative')],
    },
    'iot': {
        'temperature_c': [(78.0, 'spike'), (-15.0, 'spike'), (95.0, 'critical')],
        'power_kw':      [(~375, 'correlated_spike')],   # 2.5x normal
    }
}
print("Sample datasets loaded.")
print(f"Retail: {retail.shape} | IoT: {iot.shape}")


Sample datasets loaded.
Retail: (500, 10) | IoT: (8760, 10)


## Method 1: Z-Score Detection

In [10]:
def zscore_detect(series: pl.Series, threshold: float = 3.0) -> pl.DataFrame:
    vals = series.drop_nulls()
    mean = vals.mean()
    std = vals.std()

    values = series.to_list()
    z_scores = []
    is_anomaly = []
    for value in values:
        if value is None:
            z_scores.append(None)
            is_anomaly.append(False)
        else:
            z_score = abs((value - mean) / std) if std not in (None, 0) else 0.0
            z_scores.append(z_score)
            is_anomaly.append(z_score >= threshold)

    return pl.DataFrame({
        'value': values,
        'z_score': z_scores,
        'is_anomaly': is_anomaly,
        'method': ['zscore'] * len(values),
    }).filter(pl.col('is_anomaly'))

for col in ['temperature_c', 'power_kw', 'vibration_mm_s']:
    anomalies = zscore_detect(iot[col])
    print(f"Z-Score ({col}): {len(anomalies)} anomalies detected "
          f"(threshold=3.0, out of {iot[col].drop_nulls().len()} non-null rows)")
    if len(anomalies) > 0:
        print(f"  Top 3 by |z|: {anomalies.sort('z_score', descending=True)['value'].head(3).to_list()}")


Z-Score (temperature_c): 3 anomalies detected (threshold=3.0, out of 8585 non-null rows)
  Top 3 by |z|: [95.0, 78.0, -15.0]
Z-Score (power_kw): 1 anomalies detected (threshold=3.0, out of 8585 non-null rows)
  Top 3 by |z|: [407.25]
Z-Score (vibration_mm_s): 0 anomalies detected (threshold=3.0, out of 8585 non-null rows)


## Method 2: IQR / Tukey Fence Detection

In [11]:
def iqr_detect(series: pl.Series, multiplier: float = 1.5) -> pl.DataFrame:
    vals = series.drop_nulls()
    q1 = vals.quantile(0.25)
    q3 = vals.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr

    values = series.to_list()
    is_anomaly = []
    direction = []
    for value in values:
        if value is None:
            is_anomaly.append(False)
            direction.append('normal')
        elif value < lower:
            is_anomaly.append(True)
            direction.append('low')
        elif value > upper:
            is_anomaly.append(True)
            direction.append('high')
        else:
            is_anomaly.append(False)
            direction.append('normal')

    return pl.DataFrame({
        'value': values,
        'is_anomaly': is_anomaly,
        'direction': direction,
        'lower_fence': [lower] * len(values),
        'upper_fence': [upper] * len(values),
    }).filter(pl.col('is_anomaly'))

for mult in [1.5, 3.0]:
    anomalies = iqr_detect(iot['temperature_c'], mult)
    print(f"IQR multiplier={mult}: {len(anomalies)} anomalies in temperature_c")


IQR multiplier=1.5: 3 anomalies in temperature_c
IQR multiplier=3.0: 2 anomalies in temperature_c


## Method 3: Consensus Detection — both methods must agree

In [12]:
def consensus_detect(series: pl.Series, z_thresh=3.0, iqr_mult=1.5):
    vals = series.drop_nulls()
    mean, std = vals.mean(), vals.std()
    q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
    iqr = q3 - q1

    values = series.to_list()
    z_scores = []
    z_flag = []
    iqr_flag = []
    consensus = []
    severity = []

    for value in values:
        if value is None:
            z_scores.append(None)
            z_flag.append(False)
            iqr_flag.append(False)
            consensus.append(False)
            severity.append('normal')
            continue

        z_score = abs((value - mean) / std) if std not in (None, 0) else 0.0
        z_scores.append(z_score)
        zf = z_score >= z_thresh
        iqrf = value < q1 - iqr_mult * iqr or value > q3 + iqr_mult * iqr
        cons = zf and iqrf
        z_flag.append(zf)
        iqr_flag.append(iqrf)
        consensus.append(cons)
        if cons and z_score >= 5.0:
            severity.append('critical')
        elif cons:
            severity.append('high')
        elif zf:
            severity.append('medium')
        elif iqrf:
            severity.append('low')
        else:
            severity.append('normal')

    return pl.DataFrame({
        'value': values,
        'z_flag': z_flag,
        'iqr_flag': iqr_flag,
        'consensus': consensus,
        'severity': severity,
    }).filter(pl.col('severity') != 'normal')

print("── IoT temperature_c ──")
results = consensus_detect(iot['temperature_c'])
for sev in ['critical', 'high', 'medium', 'low']:
    n = results.filter(pl.col('severity') == sev).height
    vals = results.filter(pl.col('severity') == sev)['value'].to_list()[:3]
    if n:
        print(f"  {sev}: {n} anomalies — e.g. {vals}")


── IoT temperature_c ──
  critical: 3 anomalies — e.g. [78.0, -15.0, 95.0]


## False Positive Rate Analysis

In [13]:
# On a normally distributed column with no real anomalies,
# what fraction of rows does each method flag?
np.random.seed(42)
normal_data = pl.Series('normal', np.random.normal(100, 15, 10000))

z_fp   = len(zscore_detect(normal_data)) / 10000
iqr_fp = len(iqr_detect(normal_data))    / 10000
con_fp = len(consensus_detect(normal_data).filter(pl.col('consensus'))) / 10000

print("False Positive Rates on Normally Distributed Data (n=10,000):")
print(f"  Z-Score (|z|≥3.0): {z_fp:.3%}  (theoretical: 0.270%)")
print(f"  IQR (×1.5):        {iqr_fp:.3%}  (theoretical: ~0.7%)")
print(f"  Consensus (both):  {con_fp:.3%}  (lower = fewer false positives in UI ticker)")


False Positive Rates on Normally Distributed Data (n=10,000):
  Z-Score (|z|≥3.0): 0.270%  (theoretical: 0.270%)
  IQR (×1.5):        0.830%  (theoretical: ~0.7%)
  Consensus (both):  0.270%  (lower = fewer false positives in UI ticker)


## Real-time streaming simulation — anomaly:detected event latency

In [15]:
import asyncio

class MockSio:
    def __init__(self):
        self.events = []
        self.t0 = time.perf_counter()

    async def emit(self, event, data, room=None):
        self.events.append({'event': event, 'ms': round((time.perf_counter()-self.t0)*1000,1), **data})

async def stream_anomalies(series, name):
    sio = MockSio()
    results = consensus_detect(series)
    for row in results.iter_rows(named=True):
        await sio.emit('anomaly:detected', {
            'column': name,
            'value': row['value'],
            'severity': row['severity'],
        })
    await sio.emit('anomaly:complete', {'total_count': results.height})
    return sio.events

# In Jupyter, await the coroutine directly instead of using asyncio.run()
events = await stream_anomalies(iot['temperature_c'], 'temperature_c')
firsts = [e for e in events if e['event'] == 'anomaly:detected']
done = next(e for e in events if e['event'] == 'anomaly:complete')

if firsts:
    print(f"First anomaly:detected at {firsts[0]['ms']}ms (severity={firsts[0]['severity']})")
    print(f"All anomalies streamed by {done['ms']}ms")
    print(f"Total: {done['total_count']} anomalies across {done['ms']}ms")
    print("\n→ User sees first anomaly card immediately, not after full scan completes.")


First anomaly:detected at 10.9ms (severity=critical)
All anomalies streamed by 11.0ms
Total: 3 anomalies across 11.0ms

→ User sees first anomaly card immediately, not after full scan completes.


## Retail: known negative-revenue anomaly detection

In [16]:
print("── Revenue column anomalies ──")
revenue_anomalies = consensus_detect(retail['revenue'].drop_nulls())
for row in revenue_anomalies.filter(pl.col('value') < 0).iter_rows(named=True):
    print(f"  DETECTED: revenue={row['value']}, severity={row['severity']}, "
          f"z_flag={row['z_flag']}, iqr_flag={row['iqr_flag']}")

known = [-120.5, -45.0, -890.0]
detected_vals = revenue_anomalies['value'].to_list()
recall = sum(1 for k in known if any(abs(k-d) < 1 for d in detected_vals)) / len(known)
print(f"\nRecall on known negatives: {recall:.0%} ({sum(1 for k in known if any(abs(k-d)<1 for d in detected_vals))}/{len(known)})")


── Revenue column anomalies ──
  DETECTED: revenue=-120.5, severity=low, z_flag=False, iqr_flag=True
  DETECTED: revenue=-45.0, severity=low, z_flag=False, iqr_flag=True
  DETECTED: revenue=-890.0, severity=high, z_flag=True, iqr_flag=True
  DETECTED: revenue=-120.5, severity=low, z_flag=False, iqr_flag=True

Recall on known negatives: 100% (3/3)
